# 01. Análise Exploratória de Dados (EDA)
## Tech Challenge Fase 1, Case NPS Preditivo

Esta é a fase de Data Understanding do CRISP-DM. Aqui eu respondo, com foco em
negócio, as perguntas do requisito 3: o que é mais crítico para a satisfação, o que
gera detrator, se existe um ponto de ruptura na experiência e que tipo de cliente
costuma ter NPS mais alto ou mais baixo.

O código pesado fica em `src/`, e eu reaproveito ele aqui. Assim o notebook conta a
história e os scripts garantem que tudo seja reproduzível.

In [1]:
# Localiza a pasta src/ subindo a partir do diretorio atual, para o notebook
# funcionar tanto rodando de notebooks/ quanto da raiz do projeto.
import sys
from pathlib import Path

def _find_src(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "src" / "config.py").exists():
            return cand / "src"
    raise FileNotFoundError("Pasta src/ nao encontrada a partir de " + str(p))

SRC = _find_src()
sys.path.insert(0, str(SRC))
print("src em:", SRC)

src em: C:\workspace-fiap\tech-challenge-fase1-nps-preditivo\src


In [2]:
import pandas as pd
import config
from data_preparation import load_processed
import eda

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# Carrega a base tratada e (re)gera as figuras e o resumo numerico.
df = load_processed()
summary = eda.run()
df.shape

EDA concluida. 13 figuras salvas em C:\workspace-fiap\tech-challenge-fase1-nps-preditivo\reports\figures
Resumo numerico salvo em C:\workspace-fiap\tech-challenge-fase1-nps-preditivo\reports\eda_summary.json


(2500, 22)

## 1. Visão geral da base
A base já chega limpa, sem valores ausentes e sem duplicatas.

In [3]:
print("Linhas x Colunas:", df.shape)
print("Valores ausentes :", int(df.isna().sum().sum()))
print("Linhas duplicadas:", int(df.duplicated().sum()))
df.head()

Linhas x Colunas: (2500, 22)
Valores ausentes : 0
Linhas duplicadas: 0


,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score,nps_score_int,nps_category,is_detrator
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5,7,Neutro,0
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0,2,Detrator,1
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5,5,Detrator,1
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3,6,Detrator,1
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9,6,Detrator,1


In [4]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
customer_id,2500.0,1250.50,721.83,1.00,625.75,1250.50,1875.25,2500.00
customer_age,2500.0,43.40,14.89,18.00,31.00,43.00,56.00,69.00
customer_tenure_months,2500.0,61.32,34.48,1.00,31.00,62.00,91.00,119.00
order_id,2500.0,51250.50,721.83,50001.00,50625.75,51250.50,51875.25,52500.00
order_value,2500.0,434.26,289.77,7.76,220.24,375.52,577.29,1983.81
items_quantity,2500.0,3.47,1.69,1.00,2.00,3.00,5.00,6.00
discount_value,2500.0,29.75,29.23,0.02,8.88,20.94,40.83,230.33
payment_installments,2500.0,6.00,3.16,1.00,3.00,6.00,9.00,11.00
delivery_time_days,2500.0,8.02,3.77,2.00,5.00,8.00,11.00,14.00
delivery_delay_days,2500.0,2.19,1.45,0.00,1.00,2.00,3.00,8.00


## 2. Distribuição do NPS e categorias

Como o nps_score vem com casas decimais, eu arredondo para o inteiro mais próximo e
aplico a régua clássica do NPS: 0 a 6 Detrator, 7 a 8 Neutro, 9 a 10 Promotor.

In [5]:
dist = df[config.COL_NPS_CATEGORY].value_counts().reindex(["Detrator","Neutro","Promotor"])
pct  = (dist / dist.sum() * 100).round(1)
nps_metric = pct["Promotor"] - pct["Detrator"]
print(pd.DataFrame({"clientes": dist, "%": pct}))
print(f"\nNPS (=%Promotores - %Detratores) = {nps_metric:.1f}")

              clientes     %
nps_category                
Detrator          1975  79.0
Neutro             369  14.8
Promotor           156   6.2

NPS (=%Promotores - %Detratores) = -72.8


A empresa está numa crise de satisfação: quase 4 em cada 5 clientes são detratores e
o NPS é bem negativo.

![Distribuição](../reports/figures/01_distribuicao_nps.png)
![Categorias](../reports/figures/02_categorias_nps.png)

## 3. O que se relaciona com a satisfação

Correlação de cada variável com a nota de NPS. Em cinza, as variáveis que eu excluo do
modelo por vazamento ou por serem proxy (explico no notebook 02).

In [6]:
cols = config.OPERATIONAL_NUMERIC + config.EXCLUDED_FROM_PRIMARY_MODEL
corr = (df[cols + [config.TARGET_REGRESSION]].corr(numeric_only=True)[config.TARGET_REGRESSION]
        .drop(config.TARGET_REGRESSION).sort_values())
corr.round(3)

delivery_delay_days         -0.597
complaints_count            -0.497
customer_service_contacts   -0.351
resolution_time_days        -0.191
freight_value               -0.041
customer_age                -0.010
customer_tenure_months      -0.010
delivery_time_days           0.001
items_quantity               0.011
payment_installments         0.024
discount_value               0.025
delivery_attempts            0.028
order_value                  0.037
csat_internal_score          0.564
repeat_purchase_30d          0.570
Name: nps_score, dtype: float64

Três vilões operacionais se destacam: atraso na entrega, reclamações e contatos com o
atendimento. Variáveis de perfil e financeiras (idade, região, valor do pedido) quase
não se relacionam com o NPS.

![Correlações](../reports/figures/03_correlacoes_drivers.png)

## 4. O ponto de ruptura: atraso na entrega

A cada dia de atraso o NPS cai e a proporção de detratores dispara. A ruptura acontece
por volta de 2 a 3 dias de atraso.

In [7]:
from eda import _bucket, group_stats
b = _bucket(df, "delivery_delay_days", [-0.1,0,1,2,3,4,60], ["0","1","2","3","4","5+"])
group_stats(df, b)

,n,nps_medio,pct_detrator,pct_promotor
delivery_delay_days,,,,
0,277,6.86,42.96,25.27
1,615,5.55,65.85,10.24
2,646,4.58,82.04,2.94
3,525,3.44,93.33,0.76
4,270,2.44,97.78,0.00
5+,167,1.28,100.00,0.00


![NPS por atraso](../reports/figures/04_nps_por_atraso.png)

## 5. Reclamações e contatos com o atendimento

Nas reclamações, a virada acontece entre a primeira e a segunda. E quanto mais o
cliente precisa acionar o atendimento, pior o NPS, sinal de que ele está tendo muito
esforço para resolver.

In [8]:
b1 = _bucket(df, "complaints_count", [-0.1,0,1,2,3,60], ["0","1","2","3","4+"])
display(group_stats(df, b1))
b2 = _bucket(df, "customer_service_contacts", [-0.1,0,1,2,3,60], ["0","1","2","3","4+"])
group_stats(df, b2)

,n,nps_medio,pct_detrator,pct_promotor
complaints_count,,,,
0,23,8.52,4.35,52.17
1,122,7.77,18.03,31.15
2,277,6.05,50.90,15.16
3,507,4.91,74.75,5.92
4+,1571,3.59,91.15,2.16


,n,nps_medio,pct_detrator,pct_promotor
customer_service_contacts,,,,
0,554,5.54,63.90,13.54
1,816,4.66,76.47,6.62
2,640,4.12,84.69,3.44
3,314,3.20,91.08,1.27
4+,176,2.47,96.02,0.57


![NPS por reclamações](../reports/figures/05_nps_por_reclamacoes.png)
![NPS por contatos](../reports/figures/06_nps_por_contatos.png)

## 6. Não é a velocidade, é o atraso

Esse foi o achado mais interessante. O tempo total de entrega quase não muda o NPS
(correlação perto de zero), mas o atraso em relação ao prazo prometido é o maior vilão.
Ou seja, cumprir a promessa importa mais do que ser rápido.

In [9]:
print("corr tempo_total x NPS:", summary["delivery_time_vs_delay"]["corr_delivery_time_days"])
print("corr atraso     x NPS:", summary["delivery_time_vs_delay"]["corr_delivery_delay_days"])

corr tempo_total x NPS: 0.001
corr atraso     x NPS: -0.597


![Tempo vs atraso](../reports/figures/07_tempo_vs_atraso.png)

## 7. Cuidado com vazamento: recompra em 30 dias

O repeat_purchase_30d separa quase perfeitamente promotor de detrator, mas ele só é
conhecido até 30 dias depois do pedido. Usar isso para prever o NPS seria trapaça.

In [10]:
leak = (df.groupby("repeat_purchase_30d", observed=True)[config.COL_NPS_CATEGORY]
        .value_counts(normalize=True).unstack().fillna(0)*100).round(1)
leak

nps_category,Detrator,Neutro,Promotor
repeat_purchase_30d,,,
0,86.5,13.5,0.0
1,0.0,28.4,71.6


![Vazamento recompra](../reports/figures/08_vazamento_recompra.png)

## 8. Perfil do cliente e região

A região quase não diferencia o NPS, o que reforça que o problema é operacional e não
de público ou geografia.

![NPS por região](../reports/figures/09_nps_por_regiao.png)

## 9. Conclusões da EDA

1. A empresa está numa crise de satisfação, com cerca de 79% de detratores e NPS perto de -73.
2. Os três vilões são operacionais: atraso na entrega, reclamações e contatos com o atendimento.
3. Existe um ponto de ruptura na faixa de 2 a 3 dias de atraso: aos 2 dias já são 82% de detratores e, a partir de 3 dias, passa de 90%.
4. Cumprir o prazo pesa mais do que ser rápido.
5. O perfil do cliente quase não explica a nota.

A boa notícia é que os principais vilões são acionáveis (logística e atendimento). No
notebook 02 eu transformo isso num modelo preditivo.